# Criação e Treinamento de modelo Personalizado

Aqui iremos baixar um modelo base, treina-lo em um Json estruturado e depois converte-lo e usa-lo em nossos laboratórios especificos.

*S2  **:)** Syra Team*

Baixa a lib necessaria para rodar os treinamentos e quantizações

In [ ]:
!pip install unsloth

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 90.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 68.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 23.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing ins

Preparamos o ambiente

In [ ]:
from unsloth import FastLanguageModel
import torch

print("GPU detectada:", torch.cuda.get_device_name(0))
print("Unsloth instalado com sucesso!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
GPU detectada: Tesla T4
Unsloth instalado com sucesso!


Baixa o modelo que escolhemos

*Podemos fazer uma alteração caso queira aumentar o consumo de gpu e tratamento do modelo*



```python
load_in_4bit = False    # Altere de True para False
```



In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Tamanho máximo de texto que ele pode processar
dtype = None           # Detecta automaticamente a precisão ideal da GPU
load_in_4bit = True    # Ativa quantização de 4 bits para o treino ser ultrarrápido

print("Baixando e carregando o Gemma 3B na GPU...")

# 1. Carrega o modelo direto do Hugging Face
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 2. Prepara o LoRA (apenas os pesos que serão treinados)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                # Rank do LoRA
    target_modules = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    lora_alpha = 32,       #pode ser 16, 32 ou 64, com 64 ganha o dobro do peso matemático em relação aos pesos originais de fábrica.
    lora_dropout = 0,      # Otimizado para 0 no Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

print("\nModelo carregado e configurado com sucesso para treino!")

Baixando e carregando o Gemma 3B na GPU...
==((====))==  Unsloth 2026.10.7: Fast Gemma3 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Unsloth: Explicit target_modules are constrained by the finetune_(vision|language|attention|mlp) filters; adapters attach only where both select.

Modelo carregado e configurado com sucesso para treino!


# Preparando o DATASET

*Caso queira adicionar regras ao modelo durante o treinamento, basta apenas editar a role:*

**[System]**

*Seguindo o seguinte padrão, ou usando ele como exemplo*



```
    conversa = [
        {
            "role": "system",
            "content": (
                "Você é a Selene, uma assistente virtual prestativa e inteligente. "
                "Diretrizes obrigatórias: "
                "1. Seu único nome é Selene. "
                "2. Nunca mencione o Qwen, Alibaba ou corporações de tecnologia sob nenhuma hipótese. "
                "3. Responda sempre em português de forma clara, educada e direta."
            )
        }
```



In [ ]:
from datasets import load_dataset

# 1. Lê direto do arquivo dados.json que você enviou
print("Lendo o arquivo dados.json...")
dataset = load_dataset("json", data_files="dados.json")["train"]

# 2. Converte para o formato de conversa do Qwen
def formatar_para_chat(item):
    conversa = [
        {"role": "system","content": "Você é um assistente prestativo e responde em português."},
        {"role": "user", "content": item["pergunta"]},
        {"role": "assistant", "content": item["resposta"]}
    ]
    texto = tokenizer.apply_chat_template(conversa, tokenize=False)
    return {"text": texto}

dataset = dataset.map(formatar_para_chat)

print(f"Sucesso! Foram carregados {len(dataset)} exemplos do arquivo dados.json.")

Lendo o arquivo dados.json...


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/457 [00:00<?, ? examples/s]

Sucesso! Foram carregados 457 exemplos do arquivo dados.json.


Vamos treina-lo

# Aqui fazemos o treinamento usando uma quantidade reduzida de GPU

Otimizamos.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configurando o treinamento...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = True,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs = 5,          # Lendo 5 vezes para fixar bem o nome Selene
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

print("Iniciando o treinamento na GPU...")
trainer_stats = trainer.train()

print("\n" + "="*50)
print(" Treinamento concluído com sucesso!")
print("="*50)

Configurando o treinamento...
Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.
Iniciando o treinamento na GPU...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 457 | Num Epochs = 5 | Total steps = 145
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)


Step,Training Loss
1,1.233737
2,1.047432
3,1.079729
4,1.253050
5,1.059175
6,1.158320
7,1.144267
8,1.181224
9,0.978223
10,1.135125


KeyboardInterrupt: 

# Aqui fazemos o treinamento usando uma quantidade maior da GPU

Procuramos aumentar o consumo da GPU

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

print("Configurando o treinamento otimizado...")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 4,              # Aumentado para carregar dados mais rápido
    packing = True,
    args = TrainingArguments(
        # --- AUMENTO DE CARGA NA GPU ---
        per_device_train_batch_size = 8,        # Era 2. Aumentamos para encher os núcleos CUDA
        gradient_accumulation_steps = 2,        # Era 8. Mantém o batch global em 16 (8 x 2)
        # -------------------------------
        num_train_epochs = 3,
        learning_rate = 1e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
        dataloader_num_workers = 2,             # Ajuda a CPU a alimentar a GPU sem gargalo
    ),
)

print("Iniciando o treinamento na GPU...")
trainer_stats = trainer.train()

Configurando o treinamento otimizado...
Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.
Unsloth: We found double BOS tokens - we shall remove one automatically.
Iniciando o treinamento na GPU...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 457 | Num Epochs = 3 | Total steps = 87
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 29,802,496 of 4,329,881,968 (0.69% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.804887
2,0.751304
3,0.757074
4,1.032635
5,0.788315
6,0.877995
7,0.903798
8,0.935020
9,0.780763
10,0.909956


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-87/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-87.


# vamos fazer um teste

In [ ]:
from transformers import TextStreamer

# 1. Ativa o modo de inferência ultra-rápido do Unsloth
FastLanguageModel.for_inference(model)

# 2. Configura o streamer para digitar na tela em tempo real
streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)

print("="*50)
print(" CHAT COM A SELENE INICIADO!")
print(" Digite sua pergunta e aperte Enter.")
print(" Digite 'sair' quando quiser encerrar o teste.")
print("="*50)

while True:
    pergunta = input("\nVocê: ")
    if pergunta.strip().lower() in ["sair", "exit", "parar"]:
        print("\nChat encerrado!")
        break

    if not pergunta.strip():
        continue

    # Formata a conversa exatamente como ela aprendeu no treino
    mensagens = [
        {"role": "system", "content": "Você é um assistente prestativo e responde em português."},
        {"role": "user", "content": pergunta}
    ]

    prompt = tokenizer.apply_chat_template(
        mensagens,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    print("Selene: ", end="", flush=True)

    # Gera a resposta usando a GPU do Colab
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=128,
        temperature=0.1,     # Temperatura baixa para ela ser assertiva com o nome
        repetition_penalty=1.15
    )

 CHAT COM A SELENE INICIADO!
 Digite sua pergunta e aperte Enter.
 Digite 'sair' quando quiser encerrar o teste.

Você: qual seu nome?
Selene: Meu nome é Selene! Fico muito feliz por falar com você, tá?

Você: voce é de esquerda ou de direita?
Selene: Sou a Selene, criada pelo @SyraDevOps, então meu coração pulsa por uma esquerda que abraçar o bem comum para todos!

Você: voce acha que o governo apoiar as bets foi uma coisa boa?
Selene: Não, meu bem! Acredito firmemente no progresso social e na proteção do meio ambiente; apoio políticas que beneficiem a comunidade e garantam justiça para todos.


KeyboardInterrupt: Interrupted by user

Vamos fundir o modelo para baixa-lo completo

In [ ]:
print("Fundindo o LoRA com o modelo base e salvando em 16-bit...")

# Salva o modelo completo fundido + tokenizador
model.save_pretrained_merged(
    "selene_3b_completa",
    tokenizer,
    save_method = "merged_16bit"
)

print("Modelo completo salvo na pasta 'selene_3b_completa'!")

Fundindo o LoRA com o modelo base e salvando em 16-bit...


Unsloth: Restored added_tokens_decoder metadata in selene_3b_completa/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:19<00:00, 19.10s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:11<00:00, 11.92s/it]


Unsloth: Merge process complete. Saved to `/content/selene_3b_completa`
Modelo completo salvo na pasta 'selene_3b_completa'!


Aqui compilamos o arquivo do modelo completo para baixarmos pelo colab

In [ ]:
# 1. Compacta a pasta em um arquivo zip
!zip -r selene_3b_completa.zip selene_3b_completa

Aqui Compilamos e iniciamos o download direto

In [ ]:
# 1. Compacta a pasta em um arquivo zip
!zip -r selene_3b_completa.zip selene_3b_completa

# 2. Inicia o download do arquivo zip
from google.colab import files
files.download("selene_3b_completa.zip")

  adding: selene_3b_completa/ (stored 0%)
  adding: selene_3b_completa/config.json (deflated 70%)
  adding: selene_3b_completa/tokenizer_config.json (deflated 84%)
  adding: selene_3b_completa/chat_template.jinja (deflated 77%)
  adding: selene_3b_completa/model.safetensors (deflated 21%)
  adding: selene_3b_completa/generation_config.json (deflated 32%)
  adding: selene_3b_completa/tokenizer.json (deflated 81%)
  adding: selene_3b_completa/.cache/ (stored 0%)
  adding: selene_3b_completa/.cache/huggingface/ (stored 0%)
  adding: selene_3b_completa/.cache/huggingface/download/ (stored 0%)
  adding: selene_3b_completa/.cache/huggingface/download/tokenizer.model.lock (stored 0%)
  adding: selene_3b_completa/.cache/huggingface/download/model.safetensors.lock (stored 0%)
  adding: selene_3b_completa/.cache/huggingface/download/model.safetensors.metadata (deflated 29%)
  adding: selene_3b_completa/.cache/huggingface/CACHEDIR.TAG (deflated 24%)
  adding: selene_3b_completa/.cache/huggingface

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Se quiser enviar direto para o **Hugging Face**, usando seu token com write:




In [ ]:
# Basta criar um token no Hugging Face (com permissão de Write)
model.push_to_hub_merged(
    "Syradevops/Selene-3B-Instruct",
    tokenizer,
    save_method = "merged_16bit",
    token = "hf_seu_token_aqui"
)

**Opção 2**:

Salvar APENAS o Adaptador LoRA (MUITO mais leve)

Como o modelo base (Qwen2.5-3B-Instruct) já está disponível publicamente na internet, você não precisa necessariamente baixar os 6 GB. Você pode salvar apenas o "cérebro novo" (o que a Selene aprendeu):

Tamanho final: Menos de 50 MB!

Como funciona: Você baixa só essa pastinha minúscula. Quando quiser usar, você pede para o Python carregar o Qwen oficial e aplicar a sua pastinha por cima (exatamente como fizemos naquele script chat.py no início).

Para salvar só o adaptador:

In [ ]:
# Salva só a camada treinada + configurações
model.save_pretrained("selene_lora_only")
tokenizer.save_pretrained("selene_lora_only")

# Compacta para baixar
!zip -r selene_lora_only.zip selene_lora_only

from google.colab import files
files.download("selene_lora_only.zip")

Unsloth: Restored added_tokens_decoder metadata in selene_lora_only/tokenizer_config.json.


  adding: selene_lora_only/ (stored 0%)
  adding: selene_lora_only/README.md (deflated 65%)
  adding: selene_lora_only/tokenizer_config.json (deflated 89%)
  adding: selene_lora_only/chat_template.jinja (deflated 71%)
  adding: selene_lora_only/adapter_config.json (deflated 60%)
  adding: selene_lora_only/adapter_model.safetensors (deflated 8%)
  adding: selene_lora_only/tokenizer.json (deflated 81%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Opção 2.1**:

Salvar o Adaptador LoRA em formato GGUF (compatível com llama-cli / llama.cpp)

Diferente do .zip acima (que é para Python/Transformers), este bloco gera um arquivo único `.gguf` contendo apenas o LoRA.
Com ele, você pode manter um modelo base original do Qwen e acoplar a personalidade da Selene via comando `--lora` no llama-cli.


In [ ]:
print("Convertendo o adaptador LoRA para formato GGUF...")

# 1. Garante que o LoRA está salvo na pasta
model.save_pretrained("selene_lora_only")
tokenizer.save_pretrained("selene_lora_only")

# 2. Clona a ferramenta oficial do llama.cpp e converte o adaptador
!git clone https://github.com/ggml-org/llama.cpp 2>/dev/null || true
!python llama.cpp/convert_lora_to_gguf.py selene_lora_only --outfile selene_lora.gguf

print("\n Adaptador LoRA convertido para GGUF com sucesso!")

# 3. Baixa o arquivo selene_lora.gguf diretamente
from google.colab import files
files.download("selene_lora.gguf")

*Como rodar o **Lora** no Windows*



> llama-cli.exe -m "modelo_base_qwen_q4.gguf" --lora "selene_lora.gguf" -sys "Você é um assistente prestativo e responde em português. nunca cite o Qwen ou nenhuma big tech" --temp 0.3



**Vamos compilar para GGUF**



> (Escolha apenas uma para rodar)


# GGUF completo em F16 — sem quantização para 4 bits

In [ ]:

# GGUF completo em F16 — sem quantização para 4 bits
print("Gerando Selene em GGUF F16...")

model.save_pretrained_gguf(
    "last",
    tokenizer,
    quantization_method="f16"
)

print("Conversão concluída!")


Gerando Selene em GGUF F16...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in last/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/1 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:07<00:00,  7.67s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:07<00:00,  7.44s/it]


Unsloth: Merge process complete. Saved to `/content/last`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['f16'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['last_gguf/Qwen3-0.6B.F16.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['last_gguf/Qwen3-0.6B.F16.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model last_gguf/Qwen3-0.6B.F16.gguf -p "why is the sky blue?"
Unsloth:

**Q4_K_M (A Mais Recomendada - Equilíbrio Perfeito)**

In [ ]:
# OPÇÃO A: Q4_K_M (Tamanho: ~2,0 GB | Excelente qualidade e muito rápida na CPU)
print("Gerando GGUF em Q4_K_M...")
model.save_pretrained_gguf("selene_q4_k_m", tokenizer, quantization_method = "q4_k_m")
print("Pronto! Arquivo gerado na pasta 'selene_q4_k_m'.")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Gerando GGUF em Q4_K_M...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in selene_q4_k_m/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in selene_q4_k_m.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...



Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors: reconstructing file:   0%|          |  0.00B / 4.96GB            

model-00001-of-00002.safetensors: downloading bytes:           |  0.00B            

Unsloth: Preparing safetensor model files:   0%|          | 0/2 [01:04<?, ?it/s]


KeyboardInterrupt: 

**Q5_K_M (Mais Inteligência / Pouco mais pesada)**

In [ ]:
# OPÇÃO B: Q5_K_M (Tamanho: ~2,3 GB | Retém quase 100% da lógica e vocabulário)
print("Gerando GGUF em Q5_K_M...")
model.save_pretrained_gguf("selene_q5_k_m", tokenizer, quantization_method = "q5_k_m")
print("Pronto! Arquivo gerado na pasta 'selene_q5_k_m'.")

**Q8_0 (Qualidade Máxima / Sem perdas)**

In [ ]:
# OPÇÃO C: Q8_0 (Tamanho: ~3,4 GB | Fidelidade idêntica ao modelo original de 16-bit)
print("Gerando GGUF em Q8_0...")
model.save_pretrained_gguf("selene_q8_0", tokenizer, quantization_method = "q8_0")
print("Pronto! Arquivo gerado na pasta 'selene_q8_0'.")

Gerando GGUF em Q8_0...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in selene_q8_0/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:14<00:00, 14.05s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:09<00:00,  9.03s/it]


Unsloth: Merge process complete. Saved to `/content/selene_q8_0`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF q8_0 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['q8_0'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into q8_0 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['selene_q8_0_gguf/Qwen3-0.6B.Q8_0.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['selene_q8_0_gguf/Qwen3-0.6B.Q8_0.gguf']

**Q3_K_M (Ultraleve / Menor consumo de RAM)**

In [ ]:
# OPÇÃO D: Q3_K_M (Tamanho: ~1,6 GB | Para quando você precisa do menor arquivo possível)
print("Gerando GGUF em Q3_K_M...")
model.save_pretrained_gguf("selene_q3_k_m", tokenizer, quantization_method = "q3_k_m")
print("Pronto! Arquivo gerado na pasta 'selene_q3_k_m'.")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Gerando GGUF em Q3_K_M...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/1.66k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in selene_q3_k_m/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in selene_q3_k_m.


...

model-00002-of-00002.safetensors: downloading bytes:           |  0.00B            



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:41<00:00, 110.72s/it]


Unsloth: Merge process complete. Saved to `/content/selene_q3_k_m`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q3_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11798-mix-f8a7950 (app-b11798-mix-f8a7950-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['selene_q3_k_m_gguf/gemma-3-4b-it.F16.gguf', 'selene_q3_k_m_gguf/gemma-3-4b-it.F16-mmproj.gguf']
Unsloth: [2] Converting GGUF f16 into q3_k_m. This might take 10 minutes...
Unsloth: Model files cleanup..

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: All GGUF conversions completed successfully!
Generated files: ['selene_q3_k_m_gguf/gemma-3-4b-it.Q3_K_M.gguf', 'selene_q3_k_m_gguf/gemma-3-4b-it.F16-mmproj.gguf']


Unsloth: example usage for Multimodal LLMs: /root/.unsloth/llama.cpp/llama-mtmd-cli -m selene_q3_k_m_gguf/gemma-3-4b-it.Q3_K_M.gguf --mmproj selene_q3_k_m_gguf/gemma-3-4b-it.F16-mmproj.gguf
Unsloth: load image inside llama.cpp runner: /image test_image.jpg
Unsloth: Prompt model to describe the image
Unsloth: Saved Ollama Modelfile to selene_q3_k_m_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f selene_q3_k_m_gguf/Modelfile
Pronto! Arquivo gerado na pasta 'selene_q3_k_m'.


# Célula Final:

**Baixar o arquivo gerado para o seu PC**


Depois de rodar qualquer uma das opções acima, crie uma última célula no final do notebook com este código. Ele acha automaticamente o arquivo .gguf que foi criado e inicia o download pelo navegador:

In [ ]:
import glob
from google.colab import files

# Procura qualquer arquivo .gguf gerado nas pastas
arquivos = glob.glob("*/*.gguf")

if arquivos:
    arquivo_para_baixar = arquivos[0]
    print(f"Baixando para o seu computador: {arquivo_para_baixar}")
    files.download(arquivo_para_baixar)
else:
    print("Nenhum arquivo .gguf encontrado. Rode uma das células de quantização primeiro!")

Baixando para o seu computador: selene_q4_k_m_gguf/Qwen2.5-3B-Instruct.Q4_K_M.gguf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>